
# Transformations vs Actions, Filtering & Joins

## Business Scenario
Revenue Assurance team needs to identify:

- High revenue customers
- International roaming heavy users
- SLA breach patterns by region

## Problem Statement
Use Spark transformations to derive filtered datasets and join with SLA regional targets.

## Business Requirements
- Demonstrate lazy transformations
- Trigger actions explicitly
- Apply business filters
- Perform joins with SLA reference dataset


In [4]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.appName("Revenue_Assurance_Team_Analysis").getOrCreate()
df = spark.read.csv("telecom_cdr_production.csv", header=True, inferSchema=True)

# Transformation (lazy)
intl_users = df.filter(col("roaming_type") == "INTERNATIONAL")

# international users count (action)
intl_users.count()

# High revenue customers filter (Transformation)
high_revenue_customers = df.filter(col("revenue") > 1000)
# Show top 5 high revenue customers (Action)
high_revenue_customers.show(5)

# sla breach identification 
# explicit transofrmation chain 
# clear transformaction vs action 
# separation 

# SLA region reference dataset
sla_reference = spark.createDataFrame(
    [("North", 250), ("South", 260), ("East", 240), ("West", 255)],
    ["region", "expected_latency_threshold"]
)

joined_df = df.join(sla_reference, on="region", how="left")

joined_df.select("region", "sla_latency_ms", "expected_latency_threshold").filter(col("sla_latency_ms") > col("expected_latency_threshold")).show(10)


+------+--------------+--------------------------+
|region|sla_latency_ms|expected_latency_threshold|
+------+--------------+--------------------------+
|  East|           359|                       240|
|  East|           477|                       240|
|  East|           399|                       240|
|  East|           269|                       240|
|  East|           361|                       240|
|  East|           382|                       240|
|  East|           492|                       240|
|  East|           476|                       240|
|  East|           486|                       240|
|  East|           429|                       240|
+------+--------------+--------------------------+
only showing top 10 rows

